In [430]:
pip install pandas streamlit scikit-learn  streamlit

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [431]:
import ast
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer






In [474]:
movies=pd.read_csv('/home/vipin/own projects/rec/data/tmdb_5000_movies.csv')
credits=pd.read_csv('/home/vipin/own projects/rec/data/tmdb_5000_credits.csv')
ratings=pd.read_csv('/home/vipin/own projects/rec/data/ratings.csv')

In [476]:
user_movie_matrix = ratings.pivot_table(
    index='userId',
    columns='movieId',
    values='rating'
)

In [477]:
user_similarity = cosine_similarity(user_movie_matrix.fillna(0))

In [478]:
def recommend_user(user_id, top_n=5):

    user_idx = user_id - 1

    similar_users = list(enumerate(user_similarity[user_idx]))
    similar_users = sorted(similar_users, key=lambda x: x[1], reverse=True)[1:6]

    recommended_movies = set()

    for sim_user, score in similar_users:
        movies_liked = user_movie_matrix.iloc[sim_user].dropna().index
        recommended_movies.update(movies_liked)

    return list(recommended_movies)[:top_n]

In [479]:
links = pd.read_csv("links.csv")  # contains movieId ↔ tmdbId

movies_df = pd.read_csv("movies.csv")

# merge
movie_map = pd.merge(links, movies_df, on="movieId")

FileNotFoundError: [Errno 2] No such file or directory: 'links.csv'

In [434]:
movies['tagline']

0                             Enter the World of Pandora.
1          At the end of the world, the adventure begins.
2                                   A Plan No One Escapes
3                                         The Legend Ends
4                    Lost in our world, found in another.
                              ...                        
4798    He didn't come looking for trouble, but troubl...
4799    A newlywed couple's honeymoon is upended by th...
4800                                                  NaN
4801                             A New Yorker in Shanghai
4802                                                  NaN
Name: tagline, Length: 4803, dtype: object

In [435]:
movies = movies.merge(credits, left_on='original_title', right_on='title')
movies.drop(columns='original_title',inplace=True)


In [436]:
movies.columns

Index(['budget', 'genres', 'homepage', 'id', 'keywords', 'original_language',
       'overview', 'popularity', 'production_companies',
       'production_countries', 'release_date', 'revenue', 'runtime',
       'spoken_languages', 'status', 'tagline', 'title_x', 'vote_average',
       'vote_count', 'movie_id', 'title_y', 'cast', 'crew'],
      dtype='object')

In [437]:
print(movies.columns)
print(movies.head(2))

Index(['budget', 'genres', 'homepage', 'id', 'keywords', 'original_language',
       'overview', 'popularity', 'production_companies',
       'production_countries', 'release_date', 'revenue', 'runtime',
       'spoken_languages', 'status', 'tagline', 'title_x', 'vote_average',
       'vote_count', 'movie_id', 'title_y', 'cast', 'crew'],
      dtype='object')
      budget                                             genres  \
0  237000000  [{"id": 28, "name": "Action"}, {"id": 12, "nam...   
1  300000000  [{"id": 12, "name": "Adventure"}, {"id": 14, "...   

                                       homepage     id  \
0                   http://www.avatarmovie.com/  19995   
1  http://disney.go.com/disneypictures/pirates/    285   

                                            keywords original_language  \
0  [{"id": 1463, "name": "culture clash"}, {"id":...                en   
1  [{"id": 270, "name": "ocean"}, {"id": 726, "na...                en   

                                      

In [438]:
movies = movies.rename(columns={'title_x': 'title'})

In [439]:
movies = movies[['movie_id','title','overview','genres','keywords','cast','crew',
                 'vote_average','vote_count','popularity']]

In [440]:
movies.isnull().sum()
movies.dropna(inplace=True)

In [441]:
movies.info()

<class 'pandas.core.frame.DataFrame'>
Index: 4544 entries, 0 to 4546
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   movie_id      4544 non-null   int64  
 1   title         4544 non-null   object 
 2   overview      4544 non-null   object 
 3   genres        4544 non-null   object 
 4   keywords      4544 non-null   object 
 5   cast          4544 non-null   object 
 6   crew          4544 non-null   object 
 7   vote_average  4544 non-null   float64
 8   vote_count    4544 non-null   int64  
 9   popularity    4544 non-null   float64
dtypes: float64(2), int64(2), object(6)
memory usage: 390.5+ KB


In [442]:
print(movies['genres'].iloc[0])


[{"id": 28, "name": "Action"}, {"id": 12, "name": "Adventure"}, {"id": 14, "name": "Fantasy"}, {"id": 878, "name": "Science Fiction"}]


In [443]:
def convert(text):
    L = []
    for i in ast.literal_eval(text):
        L.append(i['name'])
    return L

movies['genres'] = movies['genres'].apply(convert)


In [444]:
movies['keywords'] = movies['keywords'].apply(convert)

In [445]:
#extract top 3 actors only to reduce noise
def get_cast(text):
    L = []
    for i in ast.literal_eval(text)[:3]:  # take only first 3
        L.append(i['name'])
    return L

movies['cast'] = movies['cast'].apply(get_cast)

In [446]:
def get_director(text):
    L = []
    for i in ast.literal_eval(text):
        if i['job'] == 'Director':
            L.append(i['name'])
    return L

movies['crew'] = movies['crew'].apply(get_director)

In [447]:

movies['overview'] = movies['overview'].apply(lambda x: x.split())

In [448]:
movies['tags'] = movies['overview'] + movies['genres'] + movies['keywords'] + movies['cast'] + movies['crew']

In [449]:
movies['genres'] = movies['genres'].apply(lambda x: [i.replace(" ", "") for i in x])
movies['keywords'] = movies['keywords'].apply(lambda x: [i.replace(" ", "") for i in x])
movies['cast'] = movies['cast'].apply(lambda x: [i.replace(" ", "") for i in x])
movies['crew'] = movies['crew'].apply(lambda x: [i.replace(" ", "") for i in x])

In [450]:
movies['tags'] = movies['tags'].apply(lambda x: " ".join(x))

In [451]:
movies.head()

,movie_id,title,overview,genres,keywords,cast,crew,vote_average,vote_count,popularity,tags
0,19995,Avatar,"[In, the, 22nd, century,, a, paraplegic, Marin...","[Action, Adventure, Fantasy, ScienceFiction]","[cultureclash, future, spacewar, spacecolony, ...","[SamWorthington, ZoeSaldana, SigourneyWeaver]",[JamesCameron],7.2,11800,150.437577,"In the 22nd century, a paraplegic Marine is di..."
1,285,Pirates of the Caribbean: At World's End,"[Captain, Barbossa,, long, believed, to, be, d...","[Adventure, Fantasy, Action]","[ocean, drugabuse, exoticisland, eastindiatrad...","[JohnnyDepp, OrlandoBloom, KeiraKnightley]",[GoreVerbinski],6.9,4500,139.082615,"Captain Barbossa, long believed to be dead, ha..."
2,206647,Spectre,"[A, cryptic, message, from, Bond’s, past, send...","[Action, Adventure, Crime]","[spy, basedonnovel, secretagent, sequel, mi6, ...","[DanielCraig, ChristophWaltz, LéaSeydoux]",[SamMendes],6.3,4466,107.376788,A cryptic message from Bond’s past sends him o...
3,49026,The Dark Knight Rises,"[Following, the, death, of, District, Attorney...","[Action, Crime, Drama, Thriller]","[dccomics, crimefighter, terrorist, secretiden...","[ChristianBale, MichaelCaine, GaryOldman]",[ChristopherNolan],7.6,9106,112.312950,Following the death of District Attorney Harve...
4,49529,John Carter,"[John, Carter, is, a, war-weary,, former, mili...","[Action, Adventure, ScienceFiction]","[basedonnovel, mars, medallion, spacetravel, p...","[TaylorKitsch, LynnCollins, SamanthaMorton]",[AndrewStanton],6.1,2124,43.926995,"John Carter is a war-weary, former military ca..."


vectorization   

In [452]:

cv = CountVectorizer(max_features=5000, stop_words='english')

vectors = cv.fit_transform(movies['tags']).toarray()

In [453]:
similarity = cosine_similarity(vectors)

In [454]:
tfidf = TfidfVectorizer(max_features=5000, stop_words='english')

vectors = tfidf.fit_transform(movies['tags']).toarray()

In [455]:

tfidf = TfidfVectorizer(max_features=5000, stop_words='english')

vectors = tfidf.fit_transform(movies['tags']).toarray()

In [456]:
movies['title'] = movies['title'].str.lower()

In [457]:
def recommend(movie):
    movie = movie.lower()
    
    # find index
    index = movies[movies['title'] == movie].index[0]
    
    # similarity scores
    distances = similarity[index]
    
    # sort movies based on similarity
    movies_list = sorted(list(enumerate(distances)), reverse=True, key=lambda x: x[1])[1:6]
    
    # get movie names
    recommended_movies = []
    for i in movies_list:
        recommended_movies.append(movies.iloc[i[0]].title)
    
    return recommended_movies

In [458]:
print(recommend("avatar"))

['aliens', 'moonraker', 'alien', 'alien³', 'silent running']


Evaluation  

In [459]:
import pickle
import numpy as np
import pandas as pd

In [460]:
movies = pickle.load(open('movies.pkl', 'rb'))
embeddings = pickle.load(open('embeddings.pkl', 'rb'))

In [461]:
movies['title_lower'] = movies['title'].str.lower()

movie_index = {
    title: i for i, title in enumerate(movies['title_lower'])
}

In [462]:
def recommend(movie_name, top_k=10):

    movie_name = movie_name.lower()

    if movie_name not in movie_index:
        return []

    index = movie_index[movie_name]

    norms = np.linalg.norm(embeddings, axis=1)
    target_norm = np.linalg.norm(embeddings[index])

    scores = np.dot(embeddings, embeddings[index]) / (norms * target_norm + 1e-10)

    results = []

    for i, sim in enumerate(scores):

        if i == index:
            continue

        final_score = sim
        results.append((i, final_score))

    results = sorted(results, key=lambda x: x[1], reverse=True)[:top_k]

    return [i for i, _ in results]

In [463]:
def get_relevant_movies(movie_idx):

    base_genres = set(movies.iloc[movie_idx]['genres'])

    relevant = []

    for i in range(len(movies)):
        if i == movie_idx:
            continue

        genres = set(movies.iloc[i]['genres'])

        if len(base_genres.intersection(genres)) > 0:
            relevant.append(i)

    return set(relevant)

In [464]:
def precision_at_k(recommended, relevant, k=10):

    recommended = recommended[:k]

    hits = sum([1 for i in recommended if i in relevant])

    return hits / k

In [465]:
def recall_at_k(recommended, relevant, k=10):

    recommended = recommended[:k]

    hits = sum([1 for i in recommended if i in relevant])

    return hits / (len(relevant) + 1e-10)

In [466]:
def mrr(recommended, relevant):

    for rank, i in enumerate(recommended):
        if i in relevant:
            return 1 / (rank + 1)

    return 0

In [467]:

test_movies = ['ice age'
]

precisions = []
recalls = []
mrrs = []

for movie in test_movies:

    if movie not in movie_index:
        continue

    idx = movie_index[movie]

    recommended = recommend(movie, top_k=10)
    relevant = get_relevant_movies(idx)

    p = precision_at_k(recommended, relevant)
    r = recall_at_k(recommended, relevant)
    m = mrr(recommended, relevant)

    precisions.append(p)
    recalls.append(r)
    mrrs.append(m)

    print(f"\n🎬 {movie}")
    print(f"Precision@10: {p:.2f}")
    print(f"Recall@10: {r:.2f}")
    print(f"MRR: {m:.2f}")


🎬 ice age
Precision@10: 0.90
Recall@10: 0.00
MRR: 1.00


save

In [468]:
import pickle

pickle.dump(movies, open('movies.pkl','wb'))
pickle.dump(similarity, open('similarity.pkl','wb'))

In [469]:
import numpy as np

embeddings = pickle.load(open('embeddings.pkl', 'rb'))

def recommend(movie):
    movie = movie.lower()

    if movie not in movies['title_lower'].values:
        return []

    index = movies[movies['title_lower'] == movie].index[0]

    # Compute similarity manually
    scores = np.dot(embeddings, embeddings[index]) / (
        np.linalg.norm(embeddings, axis=1) * np.linalg.norm(embeddings[index])
    )

    similar_movies = sorted(
        list(enumerate(scores)),
        key=lambda x: x[1],
        reverse=True
    )[1:11]

    return [(movies.iloc[i[0]].title, i[1]) for i in similar_movies]